# Библиотеки #

In [2]:
import pandas as pd
from datasets import load_dataset
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import random
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    Trainer, 
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.preprocessing import LabelEncoder

d:\anaconda\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Загружаем RuTweetCorp ##

In [3]:
df_pos = pd.read_csv('D:\\emoji\\positive.csv', encoding='utf-8')
df_neg = pd.read_csv('D:\\emoji\\negative.csv', encoding='utf-8')


df_pos_clean = pd.DataFrame({
    'text': df_pos['ttext'],
    'emotion': 'joy'
})

df_neg_clean = pd.DataFrame({
    'text': df_neg['ttext'],
    'emotion': 'sadness'
})


rutweet_combined = pd.concat([df_pos_clean, df_neg_clean], ignore_index=True)
print(rutweet_combined.head())

                                                text emotion
0  @first_timee хоть я и школота, но поверь, у на...     joy
1  Да, все-таки он немного похож на него. Но мой ...     joy
2  RT @KatiaCheh: Ну ты идиотка) я испугалась за ...     joy
3  RT @digger2912: "Кто то в углу сидит и погибае...     joy
4  @irina_dyshkant Вот что значит страшилка :D\nН...     joy


## Загружаем CEDR ##

In [4]:
cedr = load_dataset("cedr", "main")

def map_emotion(label):
    mapping = {
        0: "joy",
        1: "sadness", 
        2: "surprise",
        3: "fear",
        4: "anger"
    }
    
    if isinstance(label, list):
        label = label[0] if label else None
    return mapping.get(label, None)


cedr_train = pd.DataFrame(cedr["train"])
cedr_test = pd.DataFrame(cedr["test"])

cedr_train["emotion"] = cedr_train["labels"].apply(map_emotion)
cedr_test["emotion"] = cedr_test["labels"].apply(map_emotion)

cedr_train_clean = cedr_train[["text", "emotion"]].dropna()
cedr_test_clean = cedr_test[["text", "emotion"]].dropna()

print(f"CEDR train: {len(cedr_train_clean)}")
print(cedr_train_clean["emotion"].value_counts())

CEDR train: 4485
emotion
joy         1548
sadness     1403
surprise     574
fear         570
anger        390
Name: count, dtype: int64


## Объединяем датасеты ##

In [6]:
combined = pd.concat([
    rutweet_combined,
    cedr_train_clean,
    cedr_test_clean
], ignore_index=True)

print(f"Общий размер: {len(combined)}")
print(combined["emotion"].value_counts())

Общий размер: 232467
emotion
joy         116806
sadness     113704
surprise       739
fear           706
anger          512
Name: count, dtype: int64


## Обучаем baseline модель ##

In [7]:
X = combined['text'].astype(str).tolist()
y = combined['emotion'].tolist()


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

# TF-IDF векторизация
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Обучение с балансировкой классов
baseline = LogisticRegression(max_iter=1000, class_weight='balanced')
baseline.fit(X_train_tfidf, y_train)

y_pred = baseline.predict(X_test_tfidf)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-score (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Train size: 185973, Test size: 46494
Accuracy: 0.7133
F1-score (weighted): 0.7261

Classification Report:
              precision    recall  f1-score   support

       anger       0.04      0.32      0.07       102
        fear       0.11      0.61      0.19       141
         joy       0.75      0.73      0.74     23362
     sadness       0.75      0.70      0.72     22741
    surprise       0.09      0.51      0.15       148

    accuracy                           0.71     46494
   macro avg       0.35      0.57      0.37     46494
weighted avg       0.74      0.71      0.73     46494



### Создаем дополнительные примеры в наши редкие классы(anger, fear, surprise) ###

In [8]:
def augment_text(text, n_variations=2):
    variations = [text]
    
    # 1. Удаление пунктуации в конце
    if text.endswith(('.', '!', '?')):
        variations.append(text[:-1])
    
    # 2. Добавление восклицательного знака для эмоциональных текстов
    if '!' not in text:
        variations.append(text + '!')
    
    # 3. Повторение последнего слова для эмфазы
    words = text.split()
    if len(words) > 1:
        last_word = words[-1]
        variations.append(text + ' ' + last_word)
    
    return list(set(variations))[:n_variations]


augmented_data = []

rare_classes = ['anger', 'fear', 'surprise']

for emotion in rare_classes:
    class_samples = combined[combined['emotion'] == emotion]
    print(f"Оригинальных {emotion}: {len(class_samples)}")
    
    for _, row in class_samples.iterrows():
        text = row['text']
        augmented_texts = augment_text(text, n_variations=2)
        
        for aug_text in augmented_texts:
            if aug_text != text:
                augmented_data.append({
                    'text': aug_text,
                    'emotion': emotion
                })

augmented_df = pd.DataFrame(augmented_data)

combined_augmented = pd.concat([combined, augmented_df], ignore_index=True)

print(f"\nРазмер после аугментации: {len(combined_augmented)}")
print(combined_augmented['emotion'].value_counts())

Оригинальных anger: 512
Оригинальных fear: 706
Оригинальных surprise: 739

Размер после аугментации: 235273
emotion
joy         116806
sadness     113704
surprise      1815
fear          1700
anger         1248
Name: count, dtype: int64


## Обучаем ruBERT-tiny ##

In [ ]:
label_encoder = LabelEncoder()
combined_augmented['label_id'] = label_encoder.fit_transform(combined_augmented['emotion'])

# Вычисляем веса для классов (чтобы модель не игнорировала редкие)
class_counts = combined_augmented['emotion'].value_counts()
class_weights = {}
for label in label_encoder.classes_:
    class_weights[label] = 1.0 / class_counts[label]
    
print("Веса классов:")
for label, weight in class_weights.items():
    print(f"  {label}: {weight:.4f}")


train_texts, val_texts, train_labels, val_labels = train_test_split(
    combined_augmented['text'].tolist(),
    combined_augmented['label_id'].tolist(),
    test_size=0.15,
    random_state=42,
    stratify=combined_augmented['label_id']
)

print(f"Train: {len(train_texts)}, Val: {len(val_texts)}")


model_name = "cointegrated/rubert-tiny2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_)
)


def tokenize_function(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

train_encodings = tokenize_function(train_texts)
val_encodings = tokenize_function(val_texts)


class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = EmotionDataset(train_encodings, train_labels)
val_dataset = EmotionDataset(val_encodings, val_labels)


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    return {"accuracy": acc, "f1": f1}


training_args = TrainingArguments(
    output_dir="./rubert_emoji_results",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Начинаем обучение...")
trainer.train()

model.save_pretrained("./emoji_rubert_model")
tokenizer.save_pretrained("./emoji_rubert_model")

import json
with open("./emoji_rubert_model/label_mapping.json", "w") as f:
    json.dump(label_encoder.classes_.tolist(), f)

print("Модель сохранена!")

Веса классов:
  anger: 0.0008
  fear: 0.0006
  joy: 0.0000
  sadness: 0.0000
  surprise: 0.0006
Train: 199982, Val: 35291


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers]

Начинаем обучение...


d:\anaconda\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
